In [1]:
import torch
import torchvision
import torchvision.transforms
import numpy
import time
import random

class ResidualBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_shortcut=False):
        super(ResidualBlock, self).__init__()
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                     stride=stride, padding=1, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(out_channels)
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                     stride=1, padding=1, bias=False)
        self.bn2 = torch.nn.BatchNorm2d(out_channels)
        self.relu = torch.nn.ReLU(inplace=True)

        self.shortcut = torch.nn.Sequential()
        if use_shortcut:
            self.shortcut = torch.nn.Sequential(
                torch.nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                stride=stride, padding=0, bias=False),
                torch.nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x
        if self.shortcut:
            identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        out = self.relu(out)
        return out

class ResNet18(torch.nn.Module):
    def __init__(self):
        super(ResNet18, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(64)
        self.relu = torch.nn.ReLU(inplace=True)

        self.layer1 = torch.nn.Sequential(
            ResidualBlock(64, 64, stride=1, use_shortcut=False),
            ResidualBlock(64, 64, stride=1, use_shortcut=False)
        )
        self.layer2 = torch.nn.Sequential(
            ResidualBlock(64, 128, stride=2, use_shortcut=True),
            ResidualBlock(128, 128, stride=1, use_shortcut=False)
        )
        self.layer3 = torch.nn.Sequential(
            ResidualBlock(128, 256, stride=2, use_shortcut=True),
            ResidualBlock(256, 256, stride=1, use_shortcut=False)
        )
        self.layer4 = torch.nn.Sequential(
            ResidualBlock(256, 512, stride=2, use_shortcut=True),
            ResidualBlock(512, 512, stride=1, use_shortcut=False)
        )

        self.avgpool = torch.nn.AvgPool2d(kernel_size=4)
        self.flatten = torch.nn.Flatten()
        self.fc = torch.nn.Linear(512, 10)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, torch.nn.Conv2d):
            torch.nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, torch.nn.Linear):
            torch.nn.init.kaiming_uniform_(m.weight, a=5 ** 0.5)
            if m.bias is not None:
                torch.nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = self.flatten(out)
        out = self.fc(out)
        return out

def train(model, optimizer, train_loader, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    num_samples = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = torch.nn.functional.cross_entropy(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        num_samples += batch_x.size(0)

    avg_loss = total_loss / num_samples
    accuracy = total_correct / num_samples * 100.0
    return avg_loss, accuracy

def test(model, test_loader, device):
    model.eval()
    total_correct = 0
    num_samples = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
            num_samples += batch_x.size(0)

    accuracy = total_correct / num_samples * 100.0
    return accuracy

if __name__ == '__main__':
    EPOCHS = 100
    BATCH_SIZE = 128
    TRAIN_ACC_THRESHOLD = 90.0

    # 首次加载数据集，目的是计算均值和标准差
    temp_train_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=True, download=True, transform=None)
    data_np = temp_train_set.data.astype(numpy.float64) / 255.0
    mean = torch.tensor(data_np.mean(axis=(0, 1, 2)), dtype=torch.float32)
    std = torch.tensor(data_np.std(axis=(0, 1, 2)), dtype=torch.float32)

    # 定义我们的正则化 transform
    train_transform = torchvision.transforms.Compose([
        torchvision.transforms.RandomCrop(32, padding=4),
        torchvision.transforms.RandomHorizontalFlip(),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean, std)
    ])
    test_transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean, std)
    ])

    # 再次加载数据集，应用我们的正则化
    train_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=True, download=True, transform=train_transform)
    test_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=False, download=True, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")

    model = ResNet18().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        train_loss, train_acc = train(model, optimizer, train_loader, device)
        epoch_time = time.time() - start_time

        print(f"Epoch {epoch:2d}: TrainTime = {epoch_time:.2f}s, "
              f"TrainLoss = {train_loss:.4f}, TrainAcc = {train_acc:.2f}%", end=' ')

        if train_acc >= TRAIN_ACC_THRESHOLD:
            test_acc = test(model, test_loader, device)
            print(f"TestAcc = {test_acc:.2f}%")
        else:
            print()

Using device: cuda
Epoch  1: TrainTime = 53.71s, TrainLoss = 1.5805, TrainAcc = 41.63% 
Epoch  2: TrainTime = 51.49s, TrainLoss = 1.1152, TrainAcc = 59.87% 
Epoch  3: TrainTime = 51.63s, TrainLoss = 0.8843, TrainAcc = 68.48% 
Epoch  4: TrainTime = 51.68s, TrainLoss = 0.7350, TrainAcc = 74.04% 
Epoch  5: TrainTime = 51.77s, TrainLoss = 0.6368, TrainAcc = 77.88% 
Epoch  6: TrainTime = 51.76s, TrainLoss = 0.5644, TrainAcc = 80.40% 
Epoch  7: TrainTime = 51.83s, TrainLoss = 0.5125, TrainAcc = 82.10% 
Epoch  8: TrainTime = 51.82s, TrainLoss = 0.4748, TrainAcc = 83.46% 
Epoch  9: TrainTime = 51.75s, TrainLoss = 0.4300, TrainAcc = 85.04% 
Epoch 10: TrainTime = 51.94s, TrainLoss = 0.4031, TrainAcc = 85.95% 
Epoch 11: TrainTime = 52.12s, TrainLoss = 0.3702, TrainAcc = 86.83% 
Epoch 12: TrainTime = 52.17s, TrainLoss = 0.3484, TrainAcc = 87.91% 
Epoch 13: TrainTime = 52.00s, TrainLoss = 0.3249, TrainAcc = 88.86% 
Epoch 14: TrainTime = 52.17s, TrainLoss = 0.3026, TrainAcc = 89.42% 
Epoch 15: Train

```bash
(base) root@VM-0-80-ubuntu:/workspace# nvidia-smi 
Tue Apr  7 00:23:34 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:09.0 Off |                    0 |
| N/A   63C    P0    35W /  70W |   2044MiB / 15360MiB |     80%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-----------------------------------------------------------------------------+
| Processes:                                                                  |
|  GPU   GI   CI        PID   Type   Process name                  GPU Memory |
|        ID   ID                                                   Usage      |
|=============================================================================|
+-----------------------------------------------------------------------------+
```